In [ ]:
import os
import re
import glob
import copy
import numpy as np
import datetime as dt
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import AutoMinorLocator
from scipy.special import jv
from lmfit import Parameters, minimize, report_fit

from JunoPy.mag import MagData, ForceFreeFluxRope, merge_magdata
from JunoPy.index import IndexFile
from JunoPy.waves import WavesData, get_waves_data, merge_wavesdata
from JunoPy.library import find_ticks_between
from JunoPy.spice import InitSpiceKernels, get_juno_position

from force_free_plot_all import plot_mva_composite

### 1. Run force free fit.

In [ ]:
# start = dt.datetime.strptime('2017-338T01:49:57', '%Y-%jT%H:%M:%S')
# end   = dt.datetime.strptime('2017-338T01:51:00', '%Y-%jT%H:%M:%S')

start = dt.datetime.strptime('2017-230T13:18:58', '%Y-%jT%H:%M:%S')
end   = dt.datetime.strptime('2017-230T13:20:24', '%Y-%jT%H:%M:%S')

# start = dt.datetime.strptime('2019-089T20:35:40', '%Y-%jT%H:%M:%S')
# end   = dt.datetime.strptime('2019-089T20:36:44', '%Y-%jT%H:%M:%S')

# start = dt.datetime.strptime('2019-089T20:16:40', '%Y-%jT%H:%M:%S')
# end   = dt.datetime.strptime('2019-089T20:16:59', '%Y-%jT%H:%M:%S')

# start = dt.datetime.strptime('2017-186T20:16:40', '%Y-%jT%H:%M:%S')
# end   = dt.datetime.strptime('2017-186T20:16:59', '%Y-%jT%H:%M:%S')

magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
FFFR = ForceFreeFluxRope(magfile, start, end)

FFFR

wavesdataE, _ = get_waves_data(start, end)

### 3. Plot the composite plot.

In [ ]:
plot_mva_composite(FFFR, wavesdataE, start, end)

In [ ]:
# start = dt.datetime.strptime('2017-338T01:49:57', '%Y-%jT%H:%M:%S')
# end   = dt.datetime.strptime('2017-338T01:51:00', '%Y-%jT%H:%M:%S')

start = dt.datetime.strptime('2017-075T23:57:09', '%Y-%jT%H:%M:%S')
end   = dt.datetime.strptime('2017-075T23:58:13', '%Y-%jT%H:%M:%S')

newday = dt.datetime.strptime('2017-076T00:00:00', '%Y-%jT%H:%M:%S')

# start = dt.datetime.strptime('2019-089T20:35:40', '%Y-%jT%H:%M:%S')
# end   = dt.datetime.strptime('2019-089T20:36:44', '%Y-%jT%H:%M:%S')

# start = dt.datetime.strptime('2019-089T20:16:40', '%Y-%jT%H:%M:%S')
# end   = dt.datetime.strptime('2019-089T20:16:59', '%Y-%jT%H:%M:%S')

# start = dt.datetime.strptime('2017-186T20:16:40', '%Y-%jT%H:%M:%S')
# end   = dt.datetime.strptime('2017-186T20:16:59', '%Y-%jT%H:%M:%S')

magfile1 = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
magfile2 = glob.glob('data/mag/*{:s}ss*.hdf5'.format((end + dt.timedelta(minutes=5)).strftime('%Y%j') ))[0]

FFFR1 = ForceFreeFluxRope(magfile1, start, end, dofit=False)
FFFR2 = ForceFreeFluxRope(magfile2, newday, newday + dt.timedelta(minutes=5), dofit=False)

FFFR = copy.copy(FFFR1)
FFFR.magdata.df = pd.concat([FFFR1.magdata.df, FFFR2.magdata.df])

wavesdataE1, _ = get_waves_data(start, end)
wavesdataE2, _ = get_waves_data(newday, newday + dt.timedelta(minutes=5))

wavesdataE = merge_wavesdata(wavesdataE1, wavesdataE2)

In [ ]:
plot_mva_composite(FFFR, wavesdataE, start, end, plasmoidtype='o-line', cmap='viridis') #, savefile=True)